In [11]:
"""
Build a single tidy CSV of Members of the European Parliament with country,
NATIONAL party (aka NationalPoliticalGroup), and EUROPEAN political group (EP group),
covering parliamentary terms 4 → 10 (1994–2029).

HOW TO USE
1) Manually download the official CSV for each parliamentary term from the
   European Parliament Open Data portal and save to an `input/` folder
   alongside this script with the following filenames:
   - meps_term4.csv  # 1994–1999
   - meps_term5.csv  # 1999–2004
   - meps_term6.csv  # 2004–2009
   - meps_term7.csv  # 2009–2014
   - meps_term8.csv  # 2014–2019
   - meps_term9.csv  # 2019–2024
   - meps_term10.csv # 2024–2029 (current)

   Dataset pages (find the CSV resource under “Resources”):https://colab.research.google.com/drive/1T06AQ63J0srGX7VdUqZBLv-DipQKWXjL
   - Term 4: https://data.europarl.europa.eu/en/datasets/members-of-the-european-parliament-meps-parliamentary-term4
   - Term 5: https://data.europarl.europa.eu/en/datasets/members-of-the-european-parliament-meps-parliamentary-term5
   - Term 6: https://data.europarl.europa.eu/en/datasets/members-of-the-european-parliament-meps-parliamentary-term6
   - Term 7: https://data.europarl.europa.eu/en/datasets/members-of-the-european-parliament-meps-parliamentary-term7
   - Term 8: https://data.europarl.europa.eu/en/datasets/members-of-the-european-parliament-meps-parliamentary-term8
   - Term 9: https://data.europarl.europa.eu/dataset/meps_9_67
   - Term 10: https://data.europarl.europa.eu/en/datasets/members-of-the-european-parliament-meps-parliamentary-term10

2) Run this script. It will:
   - Load every file it can find.
   - Auto-detect the relevant columns even if their names vary by term.
   - Normalize keys and text (trim whitespace, unify case for ISO codes).
   - Keep one row per MEP–term–affiliation period (using start/end dates when available).
   - Save `mep_database_1994_present.csv` in the `output/` folder.

OUTPUT SCHEMA
- mep_id: EP lifetime identifier (if available)
- name: Full name of the MEP
- country: Member State name
- country_code: ISO 2-letter code (if present; otherwise blank)
- national_party: National political party/label used for election
- political_group: EP political group name (historical, as listed in source)
- start_date: Start of mandate/affiliation period in the term (if present)
- end_date: End of mandate/affiliation period in the term (if present)
- term: Integer (4–10)
- term_years: String label, e.g. "1994–1999"

Notes:
• This script *does not* collapse historical group names (e.g., EPP vs EPP-ED)
  so you keep historically correct labels. You can later map them if needed.
• If some terms lack a given column (e.g., explicit start/end dates), those will
  be left blank.

MIT License — free to reuse.
"""

from __future__ import annotations
import csv
from pathlib import Path
from typing import Dict, List

import pandas as pd

# -------------------------
# Config
# -------------------------
TERM_META: Dict[int, str] = {
    4: "1994–1999",
    5: "1999–2004",
    6: "2004–2009",
    7: "2009–2014",
    8: "2014–2019",
    9: "2019–2024",
    10: "2024–2029",
}

INPUT_DIR = Path("/content/drive/MyDrive/gwenEU")
OUTPUT_DIR = Path("/content/drive/MyDrive/gwenEU")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Likely column name variants seen across terms/dumps
CANDIDATE_COLS = {
    "mep_id": [
        "id", "mep_id", "ep_id", "parliamentary_id", "PersonId", "personId"
    ],
    "name": [
        "name", "FullName", "fullName", "Name", "mepName", "lastName firstName", "familyName firstName"
    ],
    "country": [
        "Member State", "MemberState", "Country", "country", "Member State/Member"
    ],
    "country_code": [
        "Member State (Code)", "MemberStateCode", "countryCode", "country_code", "ISO2", "Member State code"
    ],
    "national_party": [
        "National Political Group", "nationalPoliticalGroup", "National party", "national_party", "National Party"
    ],
    "political_group": [
        "Political Group", "politicalGroup", "Group", "EP Group", "group"
    ],
    "start_date": [
        "Start Date", "start", "startDate", "Mandate Start", "from", "start_of_term"
    ],
    "end_date": [
        "End Date", "end", "endDate", "Mandate End", "to", "end_of_term"
    ],
}


def best_col(df: pd.DataFrame, options: List[str]) -> str | None:
    """Return the first option present in df.columns (case-insensitive)."""
    lower = {c.lower(): c for c in df.columns}
    for opt in options:
        if opt.lower() in lower:
            return lower[opt.lower()]
    return None


def load_and_standardize(path: Path, term: int) -> pd.DataFrame:
    # Try detecting delimiter & encoding
    try:
        sample = path.read_bytes()[:2048]
        import chardet
        enc = chardet.detect(sample)["encoding"] or "utf-8"
    except Exception:
        enc = "utf-8"

    try:
        df = pd.read_csv(path, sep=None, engine="python", encoding=enc)
    except Exception:
        # Retry with semicolon
        df = pd.read_csv(path, sep=";", encoding=enc)

    print(f"  Columns found in term {term}: {list(df.columns)}")

    mapping = {}
    for key, opts in CANDIDATE_COLS.items():
        col = best_col(df, opts)
        mapping[key] = col

    out = pd.DataFrame()
    out["mep_id"] = df[mapping["mep_id"]] if mapping["mep_id"] else pd.NA

    # Name reconstruction
    if mapping["name"]:
        out["name"] = df[mapping["name"]].astype(str).str.strip()
    else:
        first = best_col(df, ["firstName", "First name", "first_name"])
        last = best_col(df, ["familyName", "Last name", "last_name"])
        if first and last:
            out["name"] = (df[first].astype(str).str.strip() + " " + df[last].astype(str).str.strip()).str.strip()
        else:
            out["name"] = pd.NA

    for k in ["country", "country_code", "national_party", "political_group", "start_date", "end_date"]:
        col = mapping[k]
        out[k] = df[col] if col else pd.NA

    out["country_code"] = out["country_code"].astype(str).str.strip().str.upper()
    out["country"] = out["country"].astype(str).str.strip()
    out["national_party"] = out["national_party"].astype(str).str.strip()
    out["political_group"] = out["political_group"].astype(str).str.strip()

    out["term"] = term
    out["term_years"] = TERM_META.get(term, "")

    # Drop exact duplicates only if name or ID present
    if out["name"].notna().any() or out["mep_id"].notna().any():
        out = out.drop_duplicates(subset=["mep_id", "name", "country", "national_party", "political_group", "start_date", "end_date"])

    print(f"  Loaded {len(out)} rows for term {term}")
    return out


def main():
    frames = []
    for term in sorted(TERM_META.keys()):
        fp = INPUT_DIR / f"meps_term{term}.csv"
        if fp.exists():
            try:
                frames.append(load_and_standardize(fp, term))
            except Exception as e:
                print(f"WARNING: Failed to parse {fp}: {e}")
        else:
            print(f"SKIP: {fp} not found.")
    if not frames:
        raise SystemExit("No input files loaded.")
    all_df = pd.concat(frames, ignore_index=True)
    cols = ["mep_id", "name", "country", "country_code", "national_party", "political_group", "start_date", "end_date", "term", "term_years"]
    for c in cols:
        if c not in all_df.columns:
            all_df[c] = pd.NA
    all_df = all_df[cols]
    out_path = OUTPUT_DIR / "mep_database_1994_present.csv"
    all_df.to_csv(out_path, index=False)
    print(f"Saved → {out_path.resolve()} ({len(all_df)} rows)")

if __name__ == "__main__":
    main()


  Columns found in term 4: ['mep_identifier', 'mep_honorific_prefix', 'mep_given_name', 'mep_family_name', 'mep_official_given_name', 'mep_official_family_name', 'mep_gender', 'mep_citizenship', 'mep_place_of_birth', 'mep_birthday', 'mep_death_date', 'mep_image', 'mep_email', 'mep_homepage', 'mep_twitter', 'mep_facebook_page', 'mep_facebook_profile', 'mep_country_of_representation', 'mep_political_group', 'mep_instagram', 'mep_URI']
  Loaded 0 rows for term 4
  Columns found in term 5: ['mep_identifier', 'mep_honorific_prefix', 'mep_given_name', 'mep_family_name', 'mep_official_given_name', 'mep_official_family_name', 'mep_gender', 'mep_citizenship', 'mep_place_of_birth', 'mep_birthday', 'mep_death_date', 'mep_image', 'mep_email', 'mep_homepage', 'mep_twitter', 'mep_facebook_page', 'mep_facebook_profile', 'mep_country_of_representation', 'mep_political_group', 'mep_instagram', 'mep_URI']
  Loaded 0 rows for term 5
  Columns found in term 6: ['mep_identifier', 'mep_honorific_prefix', 'm

In [5]:
!pip install pandas chardet requests

In [12]:
"""
Build a single tidy CSV of Members of the European Parliament with:
- Country of representation
- European Parliament political group (EP group)

Covering parliamentary terms 4 → 10 (1994–2029).

YOU ALREADY DOWNLOADED THE DATA
Place the CSVs you have in an `input/` folder next to this script. The script
works whether you downloaded only the "Members" files, or both "Members" and
"Affiliations" files. It will:
  • Use Affiliations (if present) to get political group + dates, and enrich with
    name/country from Members.
  • Otherwise fall back to the Members files alone (using `mep_political_group`).

EXPECTED FILENAMES (any subset is fine):
  - meps_term4.csv,  meps_term5.csv,  …, meps_term10.csv
  - meps_affiliations_term4.csv, …, meps_affiliations_term10.csv  (optional)

OUTPUT
  - `output/mep_database_1994_present.csv`

OUTPUT SCHEMA
- mep_id
- name
- country
- country_code (if available)
- political_group
- start_date (if available; blank when only Members files are present)
- end_date   (if available; blank when only Members files are present)
- term (4–10)
- term_years (e.g., "1994–1999")

Notes
• Historical group labels are preserved as-is (e.g., EPP-ED stays EPP-ED for the years it existed).
• If some fields are missing in your downloads, they’ll be left blank rather than dropping rows.

MIT License — free to reuse.
"""

from __future__ import annotations
import csv
from pathlib import Path
from typing import Dict, List, Optional

import pandas as pd

# -------------------------
# Config
# -------------------------
TERM_META: Dict[int, str] = {
    4: "1994–1999",
    5: "1999–2004",
    6: "2004–2009",
    7: "2009–2014",
    8: "2014–2019",
    9: "2019–2024",
    10: "2024–2029",
}

INPUT_DIR = Path("/content/drive/MyDrive/gwenEU")
OUTPUT_DIR = Path("/content/drive/MyDrive/gwenEU")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Column name variants seen across EP data dumps
CANDIDATE_COLS = {
    "mep_id": [
        "mep_identifier", "identifier", "id", "mep_id", "ep_id", "parliamentary_id", "PersonId", "personId"
    ],
    "name": [
        "name", "FullName", "fullName", "Name", "mepName", "display_name"
    ],
    "given_name": ["mep_given_name", "given_name", "firstName", "first_name", "First name"],
    "family_name": ["mep_family_name", "family_name", "lastName", "last_name", "Last name"],
    "country": [
        "mep_country_of_representation", "Member State", "MemberState", "Country", "country"
    ],
    "country_code": [
        "mep_country_of_representation_code", "Member State (Code)", "MemberStateCode", "countryCode", "country_code", "ISO2"
    ],
    "political_group": [
        # Affiliations dataset
        "political_group", "EP Group", "group",
        # Members dataset (mep_ prefix)
        "mep_political_group"
    ],
    "start_date": ["start_date", "from", "Start Date", "start", "Mandate Start"],
    "end_date": ["end_date", "to", "End Date", "end", "Mandate End"],
}


def best_col(df: pd.DataFrame, options: List[str]) -> Optional[str]:
    """Return the first option present in df.columns (case-insensitive)."""
    lower = {c.lower(): c for c in df.columns}
    for opt in options:
        if opt.lower() in lower:
            return lower[opt.lower()]
    return None


def read_csv_safely(path: Path) -> pd.DataFrame:
    """Try a few sensible encoding/delimiter combos to read EP CSVs robustly."""
    # 1) Try auto delimiter with UTF-8
    try:
        return pd.read_csv(path, sep=None, engine="python", encoding="utf-8")
    except Exception:
        pass
    # 2) Try semicolon with UTF-8
    try:
        return pd.read_csv(path, sep=";", encoding="utf-8")
    except Exception:
        pass
    # 3) Try auto delimiter with Latin-1
    try:
        return pd.read_csv(path, sep=None, engine="python", encoding="latin-1")
    except Exception:
        pass
    # 4) Try semicolon with Latin-1
    return pd.read_csv(path, sep=";", encoding="latin-1")


# -------------------------
# Loaders
# -------------------------

def standardize_members(df: pd.DataFrame, term: int) -> pd.DataFrame:
    """Standardize a Members dataset (meps_termX.csv)."""
    m = {k: best_col(df, v) for k, v in CANDIDATE_COLS.items()}

    out = pd.DataFrame()
    out["mep_id"] = df[m["mep_id"]] if m["mep_id"] else pd.NA

    # Name: use single name column if present; else given + family
    if m["name"]:
        out["name"] = df[m["name"]].astype(str).str.strip()
    else:
        g = m.get("given_name")
        f = m.get("family_name")
        if g and f:
            out["name"] = (df[g].astype(str).str.strip() + " " + df[f].astype(str).str.strip()).str.strip()
        else:
            out["name"] = pd.NA

    out["country"] = df[m["country"]] if m["country"] else pd.NA
    out["country_code"] = df[m["country_code"]] if m["country_code"] else pd.NA
    out["political_group"] = df[m["political_group"]] if m["political_group"] else pd.NA

    # Members files usually don't have start/end dates
    out["start_date"] = pd.NA
    out["end_date"] = pd.NA

    out["term"] = term
    out["term_years"] = TERM_META.get(term, "")

    # Clean up
    for col in ["mep_id", "name", "country", "country_code", "political_group"]:
        out[col] = out[col].astype(str).str.strip()
    out["country_code"] = out["country_code"].str.upper()

    # Deduplicate conservative subset
    out = out.drop_duplicates(subset=["mep_id", "country", "political_group"])
    return out


def standardize_affiliations(aff: pd.DataFrame, members: Optional[pd.DataFrame], term: int) -> pd.DataFrame:
    """Standardize an Affiliations dataset (meps_affiliations_termX.csv) and enrich with Members if provided."""
    a = {k: best_col(aff, v) for k, v in CANDIDATE_COLS.items()}

    out = pd.DataFrame()
    out["mep_id"] = aff[a["mep_id"]] if a["mep_id"] else pd.NA
    out["political_group"] = aff[a["political_group"]] if a["political_group"] else pd.NA
    out["start_date"] = aff[a["start_date"]] if a["start_date"] else pd.NA
    out["end_date"] = aff[a["end_date"]] if a["end_date"] else pd.NA

    # Enrich with Members (name & country)
    if members is not None and not members.empty:
        m = {k: best_col(members, v) for k, v in CANDIDATE_COLS.items()}
        temp = pd.DataFrame()
        temp["mep_id"] = members[m["mep_id"]] if m["mep_id"] else pd.NA
        if m["name"]:
            temp["name"] = members[m["name"]].astype(str).str.strip()
        else:
            g = m.get("given_name")
            f = m.get("family_name")
            if g and f:
                temp["name"] = (members[g].astype(str).str.strip() + " " + members[f].astype(str).str.strip()).str.strip()
            else:
                temp["name"] = pd.NA
        temp["country"] = members[m["country"]] if m["country"] else pd.NA
        temp["country_code"] = members[m["country_code"]] if m["country_code"] else pd.NA

        out = out.merge(temp, on="mep_id", how="left")
    else:
        out["name"] = pd.NA
        out["country"] = pd.NA
        out["country_code"] = pd.NA

    out["term"] = term
    out["term_years"] = TERM_META.get(term, "")

    # Clean
    for col in ["mep_id", "name", "country", "country_code", "political_group"]:
        out[col] = out[col].astype(str).str.strip()
    out["country_code"] = out["country_code"].str.upper()

    out = out.drop_duplicates(subset=["mep_id", "country", "political_group", "start_date", "end_date"])
    return out


# -------------------------
# Main
# -------------------------

def main():
    frames = []

    for term in sorted(TERM_META.keys()):
        mem_fp = INPUT_DIR / f"meps_term{term}.csv"
        aff_fp = INPUT_DIR / f"meps_affiliations_term{term}.csv"

        members_df = pd.DataFrame()
        if mem_fp.exists():
            try:
                members_df = read_csv_safely(mem_fp)
                print(f"Loaded Members term {term} → {mem_fp}")
            except Exception as e:
                print(f"WARNING: Failed to read Members term {term}: {e}")

        affiliations_df = pd.DataFrame()
        if aff_fp.exists():
            try:
                affiliations_df = read_csv_safely(aff_fp)
                print(f"Loaded Affiliations term {term} → {aff_fp}")
            except Exception as e:
                print(f"WARNING: Failed to read Affiliations term {term}: {e}")

        if not members_df.empty and not affiliations_df.empty:
            frames.append(standardize_affiliations(affiliations_df, members_df, term))
        elif not members_df.empty:
            frames.append(standardize_members(members_df, term))
        elif not affiliations_df.empty:
            # We have affiliations but not members: still output political group; name/country may be blank
            frames.append(standardize_affiliations(affiliations_df, None, term))
        else:
            print(f"SKIP term {term}: no input files found in {INPUT_DIR}.")

    if not frames:
        raise SystemExit("No input files found. Place CSVs under input/ and rerun.")

    all_df = pd.concat(frames, ignore_index=True)

    # Final tidy ordering
    cols = [
        "mep_id", "name", "country", "country_code",
        "political_group", "start_date", "end_date",
        "term", "term_years"
    ]
    for c in cols:
        if c not in all_df.columns:
            all_df[c] = pd.NA
    all_df = all_df[cols]

    # Save CSV
    out_path = OUTPUT_DIR / "mep_database_1994_present.csv"
    all_df.to_csv(out_path, index=False, quoting=csv.QUOTE_MINIMAL)
    print(f"Saved → {out_path.resolve()} ({len(all_df)} rows)")


if __name__ == "__main__":
    main()

Loaded Members term 4 → /content/drive/MyDrive/gwenEU/meps_term4.csv
Loaded Members term 5 → /content/drive/MyDrive/gwenEU/meps_term5.csv
Loaded Members term 6 → /content/drive/MyDrive/gwenEU/meps_term6.csv
Loaded Members term 7 → /content/drive/MyDrive/gwenEU/meps_term7.csv
Loaded Members term 8 → /content/drive/MyDrive/gwenEU/meps_term8.csv
Loaded Members term 9 → /content/drive/MyDrive/gwenEU/meps_term9.csv
Loaded Members term 10 → /content/drive/MyDrive/gwenEU/meps_term10.csv
Saved → /content/drive/MyDrive/gwenEU/mep_database_1994_present.csv (5912 rows)


In [15]:
!pip install rapidfuzz

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 27.2 MB/s eta 0:00:00


In [17]:
import pandas as pd
from pathlib import Path
from rapidfuzz import process, fuzz

# Paths
input_file = Path("/content/drive/MyDrive/gwenEU/outputv2.csv.gzip")
mep_file = Path("/content/drive/MyDrive/gwenEU/mep_database_1994_present.csv")  # Update with your path
output_file = Path("/content/drive/MyDrive/gwenEU/outputv2_filled.csv.gzip")

# Load files
df = pd.read_csv(input_file, compression="gzip", low_memory=False)
mep_df = pd.read_csv(mep_file)

# Prepare for matching
mep_df["name_clean"] = mep_df["name"].astype(str).str.strip().str.lower()
df["detectedspeaker_clean"] = df["detectedspeaker"].astype(str).str.strip().str.lower()

# Deduplicate MEP names — keep first occurrence
mep_df_unique = mep_df.drop_duplicates(subset=["name_clean"], keep="first")

# Exact match mapping
exact_map = mep_df_unique.set_index("name_clean")[["country", "political_group"]].to_dict(orient="index")

df["speaker-country"] = df["detectedspeaker_clean"].map(lambda x: exact_map.get(x, {}).get("country"))
df["detectedPoliticalGroupEU"] = df["detectedspeaker_clean"].map(lambda x: exact_map.get(x, {}).get("political_group"))

# Fuzzy match for missing values
mep_names = mep_df_unique["name_clean"].tolist()

def fuzzy_lookup(name):
    if pd.isna(name) or name.strip() == "":
        return None, None
    match, score, idx = process.extractOne(name, mep_names, scorer=fuzz.WRatio)
    if score >= 90:
        row = mep_df_unique.iloc[idx]
        return row["country"], row["political_group"]
    return None, None

mask_missing = df["speaker-country"].isna() | df["detectedPoliticalGroupEU"].isna()

for idx in df[mask_missing].index:
    country, group = fuzzy_lookup(df.at[idx, "detectedspeaker_clean"])
    if pd.isna(df.at[idx, "speaker-country"]) and country:
        df.at[idx, "speaker-country"] = country
    if pd.isna(df.at[idx, "detectedPoliticalGroupEU"]) and group:
        df.at[idx, "detectedPoliticalGroupEU"] = group

# Save filled dataset
df.to_csv(output_file, compression="gzip", index=False)
print(f"Saved → {output_file}")


Saved → /content/drive/MyDrive/gwenEU/outputv2_filled.csv.gzip
